# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/keshav-geu/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** One row represents one content page's search performance for a specific reporting date.

**Time window:** I will analyse data from a mid-panel month (March 2026) so that the final month remains available for future evaluation. This avoids using the most recent data as part of model development.

In [10]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

In [11]:
query = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

### Features
- impressions
- clicks
- CTR
- average_position
- content_age_days

### Label / Proxy
- Trend direction (e.g. whether the page is declining) used as a proxy for content refresh priority.

### Context
- report_date
- content_hash_id
- client_hash_id

### Excluded
- Future data (months after March 2026), because it would leak information that would not be available when making the prediction.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
query = f"""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT content_hash_id) AS unique_pages
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,unique_pages
0,9841378,331437


In [14]:
query = f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

con.sql(query).df()

,start_date,end_date
0,2026-03-01,2026-03-31


In [15]:
query = f"""
SELECT *
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 1
"""

con.sql(query).df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [16]:
query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(gsc_impressions) AS impressions_present,
    COUNT(gsc_clicks) AS clicks_present,
    COUNT(gsc_avg_position) AS avg_position_present
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,impressions_present,clicks_present,avg_position_present
0,9841378,9841378,9841378,3611061


## 4. Data limits

This dataset only contains search performance metrics and cannot explain why a page gained or lost traffic. It does not capture content quality, algorithm updates, competitor actions, or user intent. Some metrics may also be unavailable for clients without connected Google Search Console or Google Analytics data, so the results should be treated as decision support rather than complete evidence.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.